In [41]:
import pandas as pd
from pathlib import Path
from imblearn.over_sampling import SMOTE
import time

print("=" * 70)
print("TARGETED SMOTE EXPERIMENT")
print("=" * 70)

BASE_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

DATASET_DIR = BASE_DIR / "dataset" / "cleaned"
RESULTS_DIR = BASE_DIR / "results"

X_train = pd.read_parquet(
    DATASET_DIR / "X_train.parquet"
)

y_train = pd.read_parquet(
    DATASET_DIR / "y_train.parquet"
).squeeze()

print("X_train shape :", X_train.shape)
print("y_train shape :", y_train.shape)

TARGETED SMOTE EXPERIMENT
X_train shape : (1591936, 30)
y_train shape : (1591936,)


In [42]:
print("=" * 70)
print("ORIGINAL TRAINING DISTRIBUTION")
print("=" * 70)

original_counts = y_train.value_counts().sort_index()

for label, count in original_counts.items():
    print(f"Class {label:2d} : {count:,}")

ORIGINAL TRAINING DISTRIBUTION
Class  0 : 1,323,620
Class  1 : 1,070
Class  2 : 102,413
Class  3 : 8,225
Class  4 : 137,593
Class  5 : 4,182
Class  6 : 4,308
Class  7 : 4,745
Class  8 : 9
Class  9 : 29
Class 10 : 1,540
Class 11 : 2,522
Class 12 : 1,142
Class 13 : 16
Class 14 : 522


In [43]:
print("=" * 70)
print("APPLYING TARGETED SMOTE")
print("=" * 70)

sampling_strategy = {
    1: 50000,
    3: 50000,
    5: 50000,
    6: 50000,
    7: 50000,
    8: 10000,
    9: 10000,
    10: 50000,
    11: 50000,
    12: 25000,
    13: 10000,
    14: 25000
}

print("Target distribution:")
for label, target in sampling_strategy.items():
    print(f"Class {label:2d} → {target:,}")

APPLYING TARGETED SMOTE
Target distribution:
Class  1 → 50,000
Class  3 → 50,000
Class  5 → 50,000
Class  6 → 50,000
Class  7 → 50,000
Class  8 → 10,000
Class  9 → 10,000
Class 10 → 50,000
Class 11 → 50,000
Class 12 → 25,000
Class 13 → 10,000
Class 14 → 25,000


In [44]:
start = time.time()

smote = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=42,
    k_neighbors=5
)

X_train_targeted, y_train_targeted = smote.fit_resample(
    X_train,
    y_train
)

elapsed = time.time() - start

print()
print("=" * 70)
print("TARGETED SMOTE COMPLETED")
print("=" * 70)

print("Time taken :", round(elapsed / 60, 2), "minutes")
print("New X shape:", X_train_targeted.shape)
print("New y shape:", y_train_targeted.shape)


TARGETED SMOTE COMPLETED
Time taken : 0.07 minutes
New X shape: (1993626, 30)
New y shape: (1993626,)


In [45]:
print("=" * 70)
print("TARGETED SMOTE CLASS DISTRIBUTION")
print("=" * 70)

targeted_counts = pd.Series(
    y_train_targeted
).value_counts().sort_index()

comparison = pd.DataFrame({
    "Original": original_counts,
    "Targeted_SMOTE": targeted_counts
})

comparison["Increase"] = (
    comparison["Targeted_SMOTE"] -
    comparison["Original"]
)

comparison["Increase_%"] = (
    comparison["Increase"] /
    comparison["Original"] * 100
)

print(comparison)

TARGETED SMOTE CLASS DISTRIBUTION
       Original  Targeted_SMOTE  Increase     Increase_%
Label                                                   
0       1323620         1323620         0       0.000000
1          1070           50000     48930    4572.897196
2        102413          102413         0       0.000000
3          8225           50000     41775     507.902736
4        137593          137593         0       0.000000
5          4182           50000     45818    1095.600191
6          4308           50000     45692    1060.631383
7          4745           50000     45255     953.740780
8             9           10000      9991  111011.111111
9            29           10000      9971   34382.758621
10         1540           50000     48460    3146.753247
11         2522           50000     47478    1882.553529
12         1142           25000     23858    2089.141856
13           16           10000      9984   62400.000000
14          522           25000     24478    4689.2720

In [46]:
targeted_X_path = (
    DATASET_DIR / "X_train_targeted_smote.parquet"
)

targeted_y_path = (
    DATASET_DIR / "y_train_targeted_smote.parquet"
)

X_train_targeted.to_parquet(
    targeted_X_path,
    index=False
)

pd.Series(
    y_train_targeted,
    name="Label"
).to_frame().to_parquet(
    targeted_y_path,
    index=False
)

print("=" * 70)
print("TARGETED SMOTE DATA SAVED")
print("=" * 70)

print(targeted_X_path)
print(targeted_y_path)

TARGETED SMOTE DATA SAVED
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/cleaned/X_train_targeted_smote.parquet
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/cleaned/y_train_targeted_smote.parquet


In [47]:
import pandas as pd
import numpy as np
import time
import joblib

from pathlib import Path

print("=" * 70)
print("LOADING TARGETED SMOTE DATA")
print("=" * 70)

BASE_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

DATASET_DIR = BASE_DIR / "dataset" / "cleaned"
RESULTS_DIR = BASE_DIR / "results"
MODELS_DIR = BASE_DIR / "models"

X_train_targeted = pd.read_parquet(
    DATASET_DIR / "X_train_targeted_smote.parquet"
)

y_train_targeted = pd.read_parquet(
    DATASET_DIR / "y_train_targeted_smote.parquet"
).squeeze()

# Test data remains completely untouched
X_test = pd.read_parquet(
    DATASET_DIR / "X_test.parquet"
)

y_test = pd.read_parquet(
    DATASET_DIR / "y_test.parquet"
).squeeze()

print("X_train_targeted :", X_train_targeted.shape)
print("y_train_targeted :", y_train_targeted.shape)
print("X_test           :", X_test.shape)
print("y_test           :", y_test.shape)

LOADING TARGETED SMOTE DATA
X_train_targeted : (1993626, 30)
y_train_targeted : (1993626,)
X_test           : (397985, 30)
y_test           : (397985,)


In [48]:
print("=" * 70)
print("DATA VALIDATION")
print("=" * 70)

print("Missing values in training:",
      X_train_targeted.isna().sum().sum())

print("Missing values in test:",
      X_test.isna().sum().sum())

print("Training classes:",
      y_train_targeted.nunique())

print("Test classes:",
      y_test.nunique())

DATA VALIDATION
Missing values in training: 0
Missing values in test: 0
Training classes: 15
Test classes: 15


In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("TRAINING TARGETED SMOTE - RANDOM FOREST CONFIG 1")
print("=" * 70)

start = time.time()

rf_targeted = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
    random_state=42
)

rf_targeted.fit(
    X_train_targeted,
    y_train_targeted
)

training_time = time.time() - start

print()
print("=" * 70)
print("RANDOM FOREST TRAINING COMPLETED")
print("=" * 70)

print(
    "Training Time :",
    round(training_time / 60, 2),
    "minutes"
)

TRAINING TARGETED SMOTE - RANDOM FOREST CONFIG 1

RANDOM FOREST TRAINING COMPLETED
Training Time : 1.22 minutes


In [50]:
print("=" * 70)
print("EVALUATING TARGETED SMOTE RANDOM FOREST")
print("=" * 70)

start = time.time()

rf_pred = rf_targeted.predict(X_test)

prediction_time = time.time() - start

rf_accuracy = accuracy_score(y_test, rf_pred)

rf_precision = precision_score(
    y_test,
    rf_pred,
    average="weighted",
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_pred,
    average="weighted",
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_pred,
    average="weighted",
    zero_division=0
)

rf_macro_f1 = f1_score(
    y_test,
    rf_pred,
    average="macro",
    zero_division=0
)

print()
print("Accuracy       :", round(rf_accuracy, 6))
print("Weighted F1    :", round(rf_f1, 6))
print("Macro F1       :", round(rf_macro_f1, 6))
print("Prediction Time:", round(prediction_time, 2), "seconds")

EVALUATING TARGETED SMOTE RANDOM FOREST

Accuracy       : 0.99658
Weighted F1    : 0.996677
Macro F1       : 0.836709
Prediction Time: 4.68 seconds


In [51]:
from sklearn.ensemble import ExtraTreesClassifier

print("=" * 70)
print("TRAINING TARGETED SMOTE - EXTRA TREES CONFIG 1")
print("=" * 70)

start = time.time()

et_targeted = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
    random_state=42
)

et_targeted.fit(
    X_train_targeted,
    y_train_targeted
)

et_training_time = time.time() - start

print()
print("=" * 70)
print("EXTRA TREES TRAINING COMPLETED")
print("=" * 70)

print(
    "Training Time :",
    round(et_training_time / 60, 2),
    "minutes"
)

TRAINING TARGETED SMOTE - EXTRA TREES CONFIG 1

EXTRA TREES TRAINING COMPLETED
Training Time : 0.46 minutes


In [52]:
print("=" * 70)
print("EVALUATING TARGETED SMOTE EXTRA TREES")
print("=" * 70)

start = time.time()

et_pred = et_targeted.predict(X_test)

et_prediction_time = time.time() - start

et_accuracy = accuracy_score(y_test, et_pred)

et_precision = precision_score(
    y_test,
    et_pred,
    average="weighted",
    zero_division=0
)

et_recall = recall_score(
    y_test,
    et_pred,
    average="weighted",
    zero_division=0
)

et_f1 = f1_score(
    y_test,
    et_pred,
    average="weighted",
    zero_division=0
)

et_macro_f1 = f1_score(
    y_test,
    et_pred,
    average="macro",
    zero_division=0
)

print()
print("Accuracy       :", round(et_accuracy, 6))
print("Weighted F1    :", round(et_f1, 6))
print("Macro F1       :", round(et_macro_f1, 6))
print("Prediction Time:", round(et_prediction_time, 2), "seconds")

EVALUATING TARGETED SMOTE EXTRA TREES

Accuracy       : 0.996615
Weighted F1    : 0.996709
Macro F1       : 0.842482
Prediction Time: 7.3 seconds


In [2]:
import time
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("✓ Imports completed")

✓ Imports completed


In [4]:
import pandas as pd
import numpy as np

BASE_DIR = "/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System"

X_train_targeted = pd.read_parquet(
    f"{BASE_DIR}/dataset/cleaned/X_train_targeted_smote.parquet"
)

y_train_targeted = pd.read_parquet(
    f"{BASE_DIR}/dataset/cleaned/y_train_targeted_smote.parquet"
).squeeze()

X_test = pd.read_parquet(
    f"{BASE_DIR}/dataset/cleaned/X_test.parquet"
)

y_test = pd.read_parquet(
    f"{BASE_DIR}/dataset/cleaned/y_test.parquet"
).squeeze()

print("X_train_targeted:", X_train_targeted.shape)
print("y_train_targeted:", y_train_targeted.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train_targeted: (1993626, 30)
y_train_targeted: (1993626,)
X_test: (397985, 30)
y_test: (397985,)


In [5]:
print("Training classes:")
print(y_train_targeted.value_counts().sort_index())

print("\nTest classes:")
print(y_test.value_counts().sort_index())

Training classes:
Label
0     1323620
1       50000
2      102413
3       50000
4      137593
5       50000
6       50000
7       50000
8       10000
9       10000
10      50000
11      50000
12      25000
13      10000
14      25000
Name: count, dtype: int64

Test classes:
Label
0     330906
1        268
2      25603
3       2056
4      34399
5       1045
6       1077
7       1187
8          2
9          7
10       385
11       631
12       285
13         4
14       130
Name: count, dtype: int64


In [6]:
import time

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

xgb_targeted = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42
)

start = time.time()

xgb_targeted.fit(
    X_train_targeted,
    y_train_targeted
)

training_time = time.time() - start

print(f"XGBoost Targeted SMOTE training completed!")
print(f"Training time: {training_time/60:.2f} minutes")

XGBoost Targeted SMOTE training completed!
Training time: 25.91 minutes


In [7]:
start = time.time()

y_pred_xgb_targeted = xgb_targeted.predict(X_test)

prediction_time = time.time() - start

accuracy = accuracy_score(y_test, y_pred_xgb_targeted)
precision = precision_score(
    y_test, y_pred_xgb_targeted,
    average="weighted",
    zero_division=0
)
recall = recall_score(
    y_test, y_pred_xgb_targeted,
    average="weighted",
    zero_division=0
)
f1 = f1_score(
    y_test, y_pred_xgb_targeted,
    average="weighted",
    zero_division=0
)
macro_f1 = f1_score(
    y_test, y_pred_xgb_targeted,
    average="macro",
    zero_division=0
)

print(f"Accuracy:       {accuracy:.6f}")
print(f"Precision:      {precision:.6f}")
print(f"Recall:         {recall:.6f}")
print(f"Weighted F1:    {f1:.6f}")
print(f"Macro F1:       {macro_f1:.6f}")
print(f"Prediction time: {prediction_time:.2f} sec")

Accuracy:       0.996653
Precision:      0.997346
Recall:         0.996653
Weighted F1:    0.996934
Macro F1:       0.820981
Prediction time: 0.74 sec


In [1]:
print("=" * 70)
print("CHECKING TARGETED SMOTE MODELS")
print("=" * 70)

print("RF :", "✓ Available" if "rf_targeted" in globals() else "✗ Missing")
print("ET :", "✓ Available" if "et_targeted" in globals() else "✗ Missing")
print("XGB:", "✓ Available" if "xgb_targeted" in globals() else "✗ Missing")

CHECKING TARGETED SMOTE MODELS
RF : ✗ Missing
ET : ✗ Missing
XGB: ✗ Missing


In [2]:
from pathlib import Path

BASE_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

MODELS_DIR = BASE_DIR / "models"

print("=" * 70)
print("FILES INSIDE MODELS FOLDER")
print("=" * 70)

for file in sorted(MODELS_DIR.rglob("*")):
    if file.is_file():
        print(file)

FILES INSIDE MODELS FOLDER
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/.gitkeep
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/class_names.pkl
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/extra_trees_model.pkl
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/label_encoder.pkl
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/lightgbm_model.pkl
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/random_forest_model.pkl
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/stacking_config.pkl
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/stacking_meta_learner.pkl
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/xgboost_model.pkl


In [3]:
DATASET_DIR = BASE_DIR / "dataset" / "cleaned"

print()
print("=" * 70)
print("TARGETED SMOTE DATASETS")
print("=" * 70)

for name in [
    "X_train_targeted_smote.parquet",
    "y_train_targeted_smote.parquet",
    "X_test.parquet",
    "y_test.parquet"
]:
    path = DATASET_DIR / name
    print(
        f"{name}:",
        "✓ EXISTS" if path.exists() else "✗ MISSING"
    )


TARGETED SMOTE DATASETS
X_train_targeted_smote.parquet: ✓ EXISTS
y_train_targeted_smote.parquet: ✓ EXISTS
X_test.parquet: ✓ EXISTS
y_test.parquet: ✓ EXISTS


In [4]:
import pandas as pd
import numpy as np
import time
import joblib

from pathlib import Path

BASE_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

DATASET_DIR = BASE_DIR / "dataset" / "cleaned"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

X_train_targeted = pd.read_parquet(
    DATASET_DIR / "X_train_targeted_smote.parquet"
)

y_train_targeted = pd.read_parquet(
    DATASET_DIR / "y_train_targeted_smote.parquet"
).squeeze()

X_test = pd.read_parquet(
    DATASET_DIR / "X_test.parquet"
)

y_test = pd.read_parquet(
    DATASET_DIR / "y_test.parquet"
).squeeze()

print("=" * 70)
print("TARGETED SMOTE DATA LOADED")
print("=" * 70)

print("X_train_targeted :", X_train_targeted.shape)
print("y_train_targeted :", y_train_targeted.shape)
print("X_test           :", X_test.shape)
print("y_test           :", y_test.shape)

TARGETED SMOTE DATA LOADED
X_train_targeted : (1993626, 30)
y_train_targeted : (1993626,)
X_test           : (397985, 30)
y_test           : (397985,)


In [5]:
TARGETED_CONFIG = {
    "smote_strategy": "targeted",
    
    "random_forest": {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": None,
        "random_state": 42
    },

    "extra_trees": {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": None,
        "random_state": 42
    },

    "xgboost": {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42
    }
}

joblib.dump(
    TARGETED_CONFIG,
    MODELS_DIR / "targeted_smote_config.pkl"
)

print("✓ Targeted SMOTE configuration saved")

✓ Targeted SMOTE configuration saved


In [6]:
from sklearn.ensemble import RandomForestClassifier

print("=" * 70)
print("TRAINING TARGETED SMOTE RANDOM FOREST")
print("=" * 70)

start = time.time()

rf_targeted = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
    random_state=42
)

rf_targeted.fit(
    X_train_targeted,
    y_train_targeted
)

rf_training_time = time.time() - start

print()
print("=" * 70)
print("RANDOM FOREST TRAINING COMPLETED")
print("=" * 70)

print(
    "Training Time :",
    round(rf_training_time / 60, 2),
    "minutes"
)

TRAINING TARGETED SMOTE RANDOM FOREST

RANDOM FOREST TRAINING COMPLETED
Training Time : 1.27 minutes


In [7]:
rf_targeted_path = MODELS_DIR / "random_forest_targeted_smote.pkl"

joblib.dump(
    rf_targeted,
    rf_targeted_path
)

print()
print("✓ Targeted Random Forest saved")
print(rf_targeted_path)


✓ Targeted Random Forest saved
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/random_forest_targeted_smote.pkl


In [8]:
print(
    "File exists:",
    rf_targeted_path.exists()
)

File exists: True


In [9]:
from sklearn.ensemble import ExtraTreesClassifier

print("=" * 70)
print("TRAINING TARGETED SMOTE EXTRA TREES")
print("=" * 70)

start = time.time()

et_targeted = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
    random_state=42
)

et_targeted.fit(
    X_train_targeted,
    y_train_targeted
)

et_training_time = time.time() - start

print()
print("=" * 70)
print("EXTRA TREES TRAINING COMPLETED")
print("=" * 70)

print(
    "Training Time :",
    round(et_training_time / 60, 2),
    "minutes"
)

TRAINING TARGETED SMOTE EXTRA TREES

EXTRA TREES TRAINING COMPLETED
Training Time : 0.42 minutes


In [10]:
et_targeted_path = MODELS_DIR / "extra_trees_targeted_smote.pkl"

joblib.dump(
    et_targeted,
    et_targeted_path
)

print()
print("✓ Targeted Extra Trees saved")
print(et_targeted_path)

print(
    "File exists:",
    et_targeted_path.exists()
)


✓ Targeted Extra Trees saved
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/extra_trees_targeted_smote.pkl
File exists: True


In [11]:
et_targeted_path = MODELS_DIR / "extra_trees_targeted_smote.pkl"

joblib.dump(
    et_targeted,
    et_targeted_path
)

print()
print("✓ Targeted Extra Trees saved")
print(et_targeted_path)

print(
    "File exists:",
    et_targeted_path.exists()
)


✓ Targeted Extra Trees saved
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/extra_trees_targeted_smote.pkl
File exists: True


In [12]:
from xgboost import XGBClassifier

print("=" * 70)
print("TRAINING TARGETED SMOTE XGBOOST")
print("=" * 70)

start = time.time()

xgb_targeted = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42
)

xgb_targeted.fit(
    X_train_targeted,
    y_train_targeted
)

xgb_training_time = time.time() - start

print()
print("=" * 70)
print("XGBOOST TRAINING COMPLETED")
print("=" * 70)

print(
    "Training Time :",
    round(xgb_training_time / 60, 2),
    "minutes"
)

TRAINING TARGETED SMOTE XGBOOST

XGBOOST TRAINING COMPLETED
Training Time : 29.55 minutes


In [13]:
xgb_targeted_path = MODELS_DIR / "xgboost_targeted_smote.pkl"

joblib.dump(
    xgb_targeted,
    xgb_targeted_path
)

print()
print("✓ Targeted XGBoost saved")
print(xgb_targeted_path)

print(
    "File exists:",
    xgb_targeted_path.exists()
)


✓ Targeted XGBoost saved
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/xgboost_targeted_smote.pkl
File exists: True


In [15]:
import joblib
import time

rf_targeted = joblib.load(
    MODELS_DIR / "random_forest_targeted_smote.pkl"
)

et_targeted = joblib.load(
    MODELS_DIR / "extra_trees_targeted_smote.pkl"
)

xgb_targeted = joblib.load(
    MODELS_DIR / "xgboost_targeted_smote.pkl"
)

print("=" * 70)
print("TARGETED SMOTE MODELS LOADED")
print("=" * 70)

print("✓ Random Forest loaded")
print("✓ Extra Trees loaded")
print("✓ XGBoost loaded")

TARGETED SMOTE MODELS LOADED
✓ Random Forest loaded
✓ Extra Trees loaded
✓ XGBoost loaded


In [16]:
print("=" * 70)
print("CREATING TARGETED SMOTE STACKING FEATURES")
print("=" * 70)

start = time.time()

print("Generating Random Forest probabilities...")
rf_test_proba = rf_targeted.predict_proba(X_test)
print("✓ Random Forest completed")

print("Generating Extra Trees probabilities...")
et_test_proba = et_targeted.predict_proba(X_test)
print("✓ Extra Trees completed")

print("Generating XGBoost probabilities...")
xgb_test_proba = xgb_targeted.predict_proba(X_test)
print("✓ XGBoost completed")

stacking_test_features = np.hstack([
    rf_test_proba,
    et_test_proba,
    xgb_test_proba
])

print()
print("Stacking test feature shape:",
      stacking_test_features.shape)

print(
    "Time taken:",
    round((time.time() - start) / 60, 2),
    "minutes"
)

CREATING TARGETED SMOTE STACKING FEATURES
Generating Random Forest probabilities...
✓ Random Forest completed
Generating Extra Trees probabilities...
✓ Extra Trees completed
Generating XGBoost probabilities...
✓ XGBoost completed

Stacking test feature shape: (397985, 45)
Time taken: 0.19 minutes


In [17]:
print("=" * 70)
print("CREATING TARGETED SMOTE TRAINING STACKING FEATURES")
print("=" * 70)

start = time.time()

print("Generating Random Forest training probabilities...")
rf_train_proba = rf_targeted.predict_proba(X_train_targeted)
print("✓ Random Forest completed")

print("Generating Extra Trees training probabilities...")
et_train_proba = et_targeted.predict_proba(X_train_targeted)
print("✓ Extra Trees completed")

print("Generating XGBoost training probabilities...")
xgb_train_proba = xgb_targeted.predict_proba(X_train_targeted)
print("✓ XGBoost completed")

stacking_train_features = np.hstack([
    rf_train_proba,
    et_train_proba,
    xgb_train_proba
])

print()
print("=" * 70)
print("TARGETED SMOTE TRAINING STACKING FEATURES CREATED")
print("=" * 70)

print(
    "Training stacking feature shape:",
    stacking_train_features.shape
)

print(
    "Time taken:",
    round((time.time() - start) / 60, 2),
    "minutes"
)

CREATING TARGETED SMOTE TRAINING STACKING FEATURES
Generating Random Forest training probabilities...
✓ Random Forest completed
Generating Extra Trees training probabilities...
✓ Extra Trees completed
Generating XGBoost training probabilities...
✓ XGBoost completed

TARGETED SMOTE TRAINING STACKING FEATURES CREATED
Training stacking feature shape: (1993626, 45)
Time taken: 0.9 minutes


In [18]:
TRAIN_STACKING_PATH = DATASET_DIR / "stacking_train_features_targeted_smote.npy"
TEST_STACKING_PATH = DATASET_DIR / "stacking_test_features_targeted_smote.npy"

np.save(
    TRAIN_STACKING_PATH,
    stacking_train_features
)

np.save(
    TEST_STACKING_PATH,
    stacking_test_features
)

print("✓ Training stacking features saved")
print(TRAIN_STACKING_PATH)

print()
print("✓ Test stacking features saved")
print(TEST_STACKING_PATH)

✓ Training stacking features saved
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/cleaned/stacking_train_features_targeted_smote.npy

✓ Test stacking features saved
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/cleaned/stacking_test_features_targeted_smote.npy


In [19]:
from sklearn.linear_model import LogisticRegression
import time
import joblib

print("=" * 70)
print("TRAINING TARGETED SMOTE STACKING META-LEARNER")
print("=" * 70)

start = time.time()

meta_learner_targeted = LogisticRegression(
    max_iter=100,
    solver="saga",
    random_state=42
)

meta_learner_targeted.fit(
    stacking_train_features,
    y_train_targeted
)

meta_training_time = time.time() - start

print()
print("=" * 70)
print("TARGETED SMOTE META-LEARNER TRAINING COMPLETED")
print("=" * 70)

print(f"Training Time : {meta_training_time / 60:.2f} minutes")

TRAINING TARGETED SMOTE STACKING META-LEARNER

TARGETED SMOTE META-LEARNER TRAINING COMPLETED
Training Time : 1.42 minutes


In [20]:
meta_targeted_path = MODELS_DIR / "stacking_meta_learner_targeted_smote.pkl"

joblib.dump(
    meta_learner_targeted,
    meta_targeted_path
)

print("✓ Targeted SMOTE Meta-Learner saved")
print(meta_targeted_path)
print("File exists:", meta_targeted_path.exists())

✓ Targeted SMOTE Meta-Learner saved
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/stacking_meta_learner_targeted_smote.pkl
File exists: True


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report
)

print("=" * 70)
print("FINAL EVALUATION - TARGETED SMOTE STACKING ENSEMBLE")
print("=" * 70)

start = time.time()

# Final predictions using the saved meta-learner
y_pred_targeted_stacking = meta_learner_targeted.predict(
    stacking_test_features
)

prediction_time = time.time() - start

# Metrics
accuracy = accuracy_score(
    y_test,
    y_pred_targeted_stacking
)

weighted_precision = precision_score(
    y_test,
    y_pred_targeted_stacking,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    y_test,
    y_pred_targeted_stacking,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    y_test,
    y_pred_targeted_stacking,
    average="weighted",
    zero_division=0
)

macro_precision = precision_score(
    y_test,
    y_pred_targeted_stacking,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_pred_targeted_stacking,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred_targeted_stacking,
    average="macro",
    zero_division=0
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred_targeted_stacking
)

print()
print("=" * 70)
print("FINAL TARGETED SMOTE STACKING RESULTS")
print("=" * 70)

print(f"Accuracy           : {accuracy:.6f}")
print(f"Weighted Precision : {weighted_precision:.6f}")
print(f"Weighted Recall    : {weighted_recall:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")
print()
print(f"Macro Precision    : {macro_precision:.6f}")
print(f"Macro Recall       : {macro_recall:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print()
print(f"Balanced Accuracy  : {balanced_accuracy:.6f}")
print(f"Balanced Accuracy  : {balanced_accuracy * 100:.4f}%")
print()
print(f"Meta Prediction Time : {prediction_time:.2f} seconds")

FINAL EVALUATION - TARGETED SMOTE STACKING ENSEMBLE

FINAL TARGETED SMOTE STACKING RESULTS
Accuracy           : 0.996618
Weighted Precision : 0.996860
Weighted Recall    : 0.996618
Weighted F1        : 0.996705

Macro Precision    : 0.866052
Macro Recall       : 0.857707
Macro F1           : 0.853395

Balanced Accuracy  : 0.857707
Balanced Accuracy  : 85.7707%

Meta Prediction Time : 0.11 seconds


In [22]:
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred_targeted_stacking,
        zero_division=0
    )
)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    330906
           1       0.61      0.86      0.71       268
           2       1.00      1.00      1.00     25603
           3       0.96      0.99      0.98      2056
           4       0.99      0.99      0.99     34399
           5       0.93      0.99      0.96      1045
           6       0.99      0.99      0.99      1077
           7       0.99      0.99      0.99      1187
           8       1.00      1.00      1.00         2
           9       1.00      0.57      0.73         7
          10       0.89      0.93      0.91       385
          11       0.91      0.96      0.93       631
          12       0.72      0.58      0.65       285
          13       0.67      0.50      0.57         4
          14       0.31      0.52      0.39       130

    accuracy                           1.00    397985
   macro avg       0.87      0.86      0.85    397985
weig

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import time

BASE_DIR = Path(
    "/srv/data/datasets/Network-Intrusion-Detection-System/"
    "Intrusion_Detection_System"
)

DATASET_DIR = BASE_DIR / "dataset" / "cleaned"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

print("✓ Project paths restored")
print("DATASET_DIR:", DATASET_DIR)
print("MODELS_DIR :", MODELS_DIR)

✓ Project paths restored
DATASET_DIR: /srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/dataset/cleaned
MODELS_DIR : /srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models


In [3]:
stacking_train_features = np.load(
    DATASET_DIR / "stacking_train_features_targeted_smote.npy"
)

stacking_test_features = np.load(
    DATASET_DIR / "stacking_test_features_targeted_smote.npy"
)

print("=" * 70)
print("VERIFYING SAVED TARGETED STACKING FEATURES")
print("=" * 70)

print("Training features:", stacking_train_features.shape)
print("Test features    :", stacking_test_features.shape)

VERIFYING SAVED TARGETED STACKING FEATURES
Training features: (1993626, 45)
Test features    : (397985, 45)


In [4]:
y_train_targeted = pd.read_parquet(
    DATASET_DIR / "y_train_targeted_smote.parquet"
).squeeze()

y_test = pd.read_parquet(
    DATASET_DIR / "y_test.parquet"
).squeeze()

print("Training labels:", y_train_targeted.shape)
print("Test labels    :", y_test.shape)

Training labels: (1993626,)
Test labels    : (397985,)


In [5]:
from sklearn.linear_model import LogisticRegression

print("=" * 70)
print("TRAINING TARGETED SMOTE STACKING META-LEARNER")
print("=" * 70)

start = time.time()

meta_learner_targeted = LogisticRegression(
    max_iter=100,
    solver="saga",
    n_jobs=-1,
    random_state=42
)

meta_learner_targeted.fit(
    stacking_train_features,
    y_train_targeted
)

meta_training_time = time.time() - start

print()
print("=" * 70)
print("TARGETED META-LEARNER TRAINING COMPLETED")
print("=" * 70)

print(
    "Training Time :",
    round(meta_training_time / 60, 2),
    "minutes"
)

TRAINING TARGETED SMOTE STACKING META-LEARNER


/opt/tljh/user/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



TARGETED META-LEARNER TRAINING COMPLETED
Training Time : 1.39 minutes


In [6]:
TARGETED_META_PATH = (
    MODELS_DIR / "stacking_meta_learner_targeted_smote.pkl"
)

joblib.dump(
    meta_learner_targeted,
    TARGETED_META_PATH
)

print()
print("=" * 70)
print("TARGETED META-LEARNER SAVED")
print("=" * 70)

print(TARGETED_META_PATH)
print("File exists:", TARGETED_META_PATH.exists())


TARGETED META-LEARNER SAVED
/srv/data/datasets/Network-Intrusion-Detection-System/Intrusion_Detection_System/models/stacking_meta_learner_targeted_smote.pkl
File exists: True


In [7]:
print("=" * 70)
print("FINAL EVALUATION - TARGETED SMOTE STACKING ENSEMBLE")
print("=" * 70)

# Load meta-learner if necessary
meta_learner_targeted = joblib.load(
    MODELS_DIR / "stacking_meta_learner_targeted_smote.pkl"
)

start = time.time()

y_pred_targeted = meta_learner_targeted.predict(
    stacking_test_features
)

prediction_time = time.time() - start

print()
print("=" * 70)
print("TARGETED STACKING PREDICTION COMPLETED")
print("=" * 70)

print("Predictions     :", len(y_pred_targeted))
print("Prediction Time :", round(prediction_time, 2), "seconds")

FINAL EVALUATION - TARGETED SMOTE STACKING ENSEMBLE

TARGETED STACKING PREDICTION COMPLETED
Predictions     : 397985
Prediction Time : 0.15 seconds


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

accuracy = accuracy_score(
    y_test,
    y_pred_targeted
)

weighted_precision = precision_score(
    y_test,
    y_pred_targeted,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    y_test,
    y_pred_targeted,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    y_test,
    y_pred_targeted,
    average="weighted",
    zero_division=0
)

macro_precision = precision_score(
    y_test,
    y_pred_targeted,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_pred_targeted,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred_targeted,
    average="macro",
    zero_division=0
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred_targeted
)

print("=" * 70)
print("FINAL TARGETED-SMOTE STACKING METRICS")
print("=" * 70)

print(f"Accuracy           : {accuracy:.6f}")
print(f"Weighted Precision : {weighted_precision:.6f}")
print(f"Weighted Recall    : {weighted_recall:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")
print(f"Macro Precision    : {macro_precision:.6f}")
print(f"Macro Recall       : {macro_recall:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.6f}")

print()
print("Percentages:")
print(f"Accuracy           : {accuracy * 100:.4f}%")
print(f"Macro F1           : {macro_f1 * 100:.4f}%")
print(f"Balanced Accuracy  : {balanced_accuracy * 100:.4f}%")

FINAL TARGETED-SMOTE STACKING METRICS
Accuracy           : 0.996618
Weighted Precision : 0.996860
Weighted Recall    : 0.996618
Weighted F1        : 0.996705
Macro Precision    : 0.866052
Macro Recall       : 0.857707
Macro F1           : 0.853395
Balanced Accuracy  : 0.857707

Percentages:
Accuracy           : 99.6618%
Macro F1           : 85.3395%
Balanced Accuracy  : 85.7707%


In [9]:
from sklearn.metrics import classification_report

targeted_report = classification_report(
    y_test,
    y_pred_targeted,
    target_names=class_names,
    zero_division=0
)

print("=" * 70)
print("TARGETED-SMOTE STACKING CLASSIFICATION REPORT")
print("=" * 70)
print(targeted_report)

NameError: name 'class_names' is not defined

In [10]:
import joblib

label_encoder = joblib.load(
    MODELS_DIR / "label_encoder.pkl"
)

class_names = label_encoder.classes_

print("=" * 70)
print("CLASS NAMES RESTORED")
print("=" * 70)

for i, name in enumerate(class_names):
    print(f"{i:2d} -> {name}")

CLASS NAMES RESTORED
 0 -> BENIGN
 1 -> Bot
 2 -> DDoS
 3 -> DoS GoldenEye
 4 -> DoS Hulk
 5 -> DoS Slowhttptest
 6 -> DoS slowloris
 7 -> FTP-Patator
 8 -> Heartbleed
 9 -> Infiltration
10 -> PortScan
11 -> SSH-Patator
12 -> Web Attack � Brute Force
13 -> Web Attack � Sql Injection
14 -> Web Attack � XSS


In [11]:
from sklearn.metrics import classification_report

targeted_report = classification_report(
    y_test,
    y_pred_targeted,
    target_names=class_names,
    zero_division=0
)

print("=" * 70)
print("TARGETED-SMOTE STACKING CLASSIFICATION REPORT")
print("=" * 70)

print(targeted_report)

TARGETED-SMOTE STACKING CLASSIFICATION REPORT
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    330906
                       Bot       0.61      0.86      0.71       268
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       0.96      0.99      0.98      2056
                  DoS Hulk       0.99      0.99      0.99     34399
          DoS Slowhttptest       0.93      0.99      0.96      1045
             DoS slowloris       0.99      0.99      0.99      1077
               FTP-Patator       0.99      0.99      0.99      1187
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.57      0.73         7
                  PortScan       0.89      0.93      0.91       385
               SSH-Patator       0.91      0.96      0.93       631
  Web Attack � Brute Force       0.72      0.58      0.65       285
W